# 第5週 AI 分類入門與成果發表

**講師解答版：請勿於練習開始前發給學員。**

每週180分鐘，含10分鐘休息。每個W標號對應PPT與Word同一單元。

教材對照：CH11PPT 11-1至11-3；CH15PPT 15-2概念；CH16PPT 16-1流程。色彩最近中心分類器為本課新增簡化實作

學習目標：
- 能區分規則、訓練與推論
- 能分開訓練資料及保留測試資料
- 能完成分類、評估並解釋限制

先按Shift+Enter逐格執行；交件前重新啟動核心並全部執行。練習格預設以註解保存，請取消註解後完成。請勿將未完成的函數覆蓋示範函數後繼續執行。

## 課前套件
由講師先依總覽安裝套件。核心程式不下載模型或開啟攝影機；色卡由程式自行產生。輸出檔案寫在目前工作目錄的outputs資料夾。

In [1]:
import sys
print("Python", sys.version.split()[0])

Python 3.12.14


## W5-01 AI 概念與本週成果

000–010 分鐘，10分鐘。

- AI 是廣泛領域
- 機器學習從資料估計參數
- 深度學習是機器學習的一支

**操作任務**：說明手寫閾值與從樣本估計平均中心的差別。

**參考答案與驗收**：前者參數由人設定，後者中心由帶標籤訓練資料計算。

## W5-02 特徵與標籤

010–025 分鐘，15分鐘。

- 特徵描述輸入資料
- 標籤是已知類別
- 今日特徵為平均R、G、B

**操作任務**：執行資料準備，印出第一張色卡與特徵，分辨影像、特徵、標籤。

In [2]:
import numpy as np
from PIL import Image
from IPython.display import display
from pathlib import Path
import csv

def make_card(rgb_values):
    image = np.zeros((24, 24, 3), dtype=np.uint8)
    image[:] = rgb_values
    return image

def features(image_rgb):
    return image_rgb.astype(float).mean(axis=(0, 1))

card = make_card([220, 30, 30])
display(Image.fromarray(card))
print(card.shape, features(card))

(24, 24, 3) [220.  30.  30.]


### 練習
先在下方寫出預測，再取消程式註解並完成。

預測：＿＿＿＿

執行後發現：＿＿＿＿

In [3]:
my_card = make_card([30, 220, 30])
assert np.allclose(features(my_card), [30, 220, 30])
print(features(my_card))

[ 30. 220.  30.]


**參考答案與驗收**：影像(24,24,3)；特徵(3,)；標籤為red等類別文字。

## W5-03 訓練集與保留測試集

025–045 分鐘，20分鐘。

- 訓練集用來計算中心
- 測試集保留到模型固定後
- 同張圖片重複放兩邊會洩漏答案

**操作任務**：數出各類訓練與測試筆數，確認沒有相同色卡同時出現在兩側。

In [4]:
train_values = [[220,30,30], [180,20,20], [240,50,50],
                [30,220,30], [20,180,20], [50,240,50],
                [30,30,220], [20,20,180], [50,50,240]]
y_train = np.array(["red"] * 3 + ["green"] * 3 + ["blue"] * 3)
test_values = [[200,25,25], [230,40,40], [25,200,25],
               [40,230,40], [25,25,200], [40,40,230]]
y_test = np.array(["red", "red", "green", "green", "blue", "blue"])
X_train = np.array([features(make_card(v)) for v in train_values])
X_test = np.array([features(make_card(v)) for v in test_values])
print(X_train.shape, X_test.shape)

(9, 3) (6, 3)


### 練習
先在下方寫出預測，再取消程式註解並完成。

預測：＿＿＿＿

執行後發現：＿＿＿＿

In [5]:
assert len(train_values) == 9 and len(test_values) == 6
assert not (set(map(tuple, train_values)) & set(map(tuple, test_values)))
print("沒有完全重複的色卡")

沒有完全重複的色卡


**參考答案與驗收**：每類訓練3筆、測試2筆，共9與6筆。

## W5-04 訓練最近中心分類器

045–060 分鐘，15分鐘。

- 同類特徵取平均
- 中心是從訓練資料算出的參數
- 三個類別各有一個中心

**操作任務**：計算三個中心，手算紅色中心第一個通道與程式核對。

In [6]:
classes = np.unique(y_train)
centroids = []
for label in classes:
    class_rows = X_train[y_train == label]
    centroids.append(class_rows.mean(axis=0))
centroids = np.array(centroids)
for label, center in zip(classes, centroids):
    print(label, np.round(center, 2))

blue [ 33.33  33.33 213.33]
green [ 33.33 213.33  33.33]
red [213.33  33.33  33.33]


### 練習
先在下方寫出預測，再取消程式註解並完成。

預測：＿＿＿＿

執行後發現：＿＿＿＿

In [7]:
manual_red_r = (220 + 180 + 240) / 3
red_index = list(classes).index("red")
assert np.isclose(centroids[red_index, 0], manual_red_r)
print(manual_red_r)

213.33333333333334


**參考答案與驗收**：紅色中心約[213.33,33.33,33.33]，共3個中心。

## W5-05 休息與模型保存

060–070 分鐘，10分鐘。

- 休息10分鐘
- 不要用測試資料重算中心
- 保留資料分組

**操作任務**：存檔與休息。

**參考答案與驗收**：中心只由X_train與y_train計算。

## W5-06 推論與距離

070–090 分鐘，20分鐘。

- 新特徵與各中心比較
- 平方差總和最小者勝出
- 距離不是機率

**操作任務**：預測[200,25,25]，查看三個距離與最小值索引。

In [8]:
def predict_one(feature):
    distances = []
    for center in centroids:
        distances.append(float(np.sum((feature - center) ** 2)))
    index = int(np.argmin(distances))
    return str(classes[index]), distances

prediction, distances = predict_one(X_test[0])
print(prediction, np.round(distances, 2))

red [63316.67 63316.67   316.67]


### 練習
先在下方寫出預測，再取消程式註解並完成。

預測：＿＿＿＿

執行後發現：＿＿＿＿

In [9]:
prediction, _ = predict_one(features(make_card([20, 20, 210])))
assert prediction == "blue"
print(prediction)

blue


**參考答案與驗收**：預測red；最小距離所對應的類別是red。

## W5-07 測試與正確率

090–110 分鐘，20分鐘。

- 每筆預測對照已知標籤
- 正確率是答對筆數除以總筆數
- 小而簡單的測試無法代表所有情況

**操作任務**：列出六筆結果，計算correct、total與accuracy。

In [10]:
predictions = []
for feature in X_test:
    label, _ = predict_one(feature)
    predictions.append(label)
predictions = np.array(predictions)
correct = int(np.sum(predictions == y_test))
accuracy = correct / len(y_test)
for truth, pred in zip(y_test, predictions):
    print("真實", truth, "預測", pred)
print(correct, len(y_test), accuracy)

真實 red 預測 red
真實 red 預測 red
真實 green 預測 green
真實 green 預測 green
真實 blue 預測 blue
真實 blue 預測 blue
6 6 1.0


### 練習
先在下方寫出預測，再取消程式註解並完成。

預測：＿＿＿＿

執行後發現：＿＿＿＿

In [11]:
example_accuracy = 4 / 6
assert np.isclose(example_accuracy, 2 / 3)
print(f"示例正確率 {example_accuracy:.2%}")

示例正確率 66.67%


**參考答案與驗收**：六筆預期皆正確；6/6=1.0，此結果僅適用這組色卡。

## W5-08 陌生樣本與失敗案例

110–125 分鐘，15分鐘。

- 模型只認得既定三類
- 灰色也會被迫選一類
- 同距離時argmin選第一個

**操作任務**：預測[128,128,128]並觀察距離；記錄它應屬未學過的灰色。

In [12]:
unknown = make_card([128, 128, 128])
display(Image.fromarray(unknown))
unknown_label, unknown_distances = predict_one(features(unknown))
print("灰色被分為", unknown_label)
print(np.round(unknown_distances, 2))

灰色被分為 blue
[25205.33 25205.33 25205.33]


### 練習
先在下方寫出預測，再取消程式註解並完成。

預測：＿＿＿＿

執行後發現：＿＿＿＿

In [13]:
yellow_label, yellow_distances = predict_one(features(make_card([230, 230, 20])))
print(yellow_label, np.round(yellow_distances, 2))
print("黃色不在訓練標籤內，輸出不能視為正確辨識黃色")

green [114733.33  39133.33  39133.33]
黃色不在訓練標籤內，輸出不能視為正確辨識黃色


**參考答案與驗收**：距離相同，classes排序第一個為blue；這是平手規則，不是灰色理解。

## W5-09 教材 AI 應用對照

125–140 分鐘，15分鐘。

- 人臉偵測找位置，識別判身分
- 手勢可先估關鍵點再套規則
- 預訓練模型仍需要檢查結果

**操作任務**：將「找出臉框」「認出小安」「紅色卡分類」配對成偵測、識別、分類。

**參考答案與驗收**：找臉框是偵測，認出小安是身分識別，紅色卡是分類。

## W5-10 期末專題與成果發表

140–160 分鐘，20分鐘。

- 保存每筆真實標籤與預測
- 報告6筆保留測試結果
- 展示至少1個失敗或陌生案例

**操作任務**：保存predictions.csv及模型中心；完成3句結論，兩組互講或每組90秒發表。

In [14]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)
print("請整理本週成果")

請整理本週成果


### 練習
先在下方寫出預測，再取消程式註解並完成。

預測：＿＿＿＿

執行後發現：＿＿＿＿

In [15]:
with (output_dir / "predictions.csv").open("w", encoding="utf-8-sig", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["truth", "prediction"])
    writer.writerows(zip(y_test, predictions))
np.savez(output_dir / "color_model.npz", classes=classes, centroids=centroids)
conclusion = f"保留測試{len(y_test)}筆，答對{correct}筆。人工色卡不代表真實物體。灰色未在訓練類別內，仍被迫分類。"
(output_dir / "conclusion.txt").write_text(conclusion, encoding="utf-8")
print(conclusion)

保留測試6筆，答對6筆。人工色卡不代表真實物體。灰色未在訓練類別內，仍被迫分類。


**參考答案與驗收**：CSV有6筆結果；中心以npz保存；結論包含6/6、人工色卡限制、灰色被迫分類。

## W5-11 專題評量與重跑

160–170 分鐘，10分鐘。

- 程式可從乾淨核心重跑
- 資料與方法說明完整
- 限制說明與分數同樣重要

**操作任務**：依總覽評量表互評，提交Notebook、CSV與結論。

**參考答案與驗收**：可重跑且輸出完整；結論沒有將距離稱為機率，也沒有宣稱通用物體辨識。

## W5-12 五週回顧與下一步

170–180 分鐘，10分鐘。

- 變數、控制、函數、資料、影像
- 模型品質需要資料與評估
- 延伸閱讀教材12至16章

**操作任務**：離堂回答：訓練與推論差別、為何分開測試、灰色案例的限制。

**參考答案與驗收**：訓練估計中心，推論使用固定中心；保留測試檢查未用來訓練的資料；模型不能自動拒絕未知類別。

## 課後作業參考解
以下為獨立解答，請在學員完成作業後提供。

In [16]:
print("訓練用資料計算中心；推論以固定中心預測。")
print("保留測試避免用訓練資料冒充未見資料的表現。")
print("灰色不在已知標籤內，分類器仍被迫挑選一類。")

訓練用資料計算中心；推論以固定中心預測。
保留測試避免用訓練資料冒充未見資料的表現。
灰色不在已知標籤內，分類器仍被迫挑選一類。
